# Day 4 — Unity Catalog: Fundamentals & Governed Access

Allow about 2.5–3.5 hours for this notebook (plus any lecture time). Work through the sections in order, or split blocks across the class if needed.

You will use the three-level name (`catalog.schema.object`), create a personal schema and several views over existing Day 3 Delta paths on ABFS, run metadata and analytics queries, and finish with challenges and SQL drills.

Prerequisites: Unity Catalog enabled, Day 3 Silver/Gold paths available, UC-compatible compute, and `%run ./02-Mount-Azure-Data-Lake`.

Related reference material (external courses, adapted here to ABFS and views): Databricks RBAC / lineage labs (M-7), workspace permissions (M-1), and security/governance examples (e.g. masked views).

## 1 — Connect ADLS (OAuth + `base_path`)

Same pattern as Days 1–3. Run the mount helper, then set paths.

In [0]:
%run ./02-Mount-Azure-Data-Lake

# Azure Data Lake Storage Gen2 — ABFS + OAuth (no DBFS mount)

This notebook configures **Spark** to read/write Azure Data Lake Storage Gen2 using the **ABFS** URI scheme (`abfss://...`) and **OAuth** via `spark.conf` (Service Principal). **We do not use `dbutils.fs.mount`** here.

---

## Prerequisites

1. ADLS Gen2 storage account and container with your data.
2. Service Principal with **Storage Blob Data Contributor** (or equivalent) on the container.
3. Fill in **tenant_id**, **client_id**, **client_secret** below (or use `dbutils.secrets.get` for production).

**Expected data layout** under the container (folder `data/`):
- `data/json/2015-summary.json`
- `data/2010-summary.csv`

After this runs, `base_path` points at `abfss://<container>@<account>.dfs.core.windows.net/data`.

---

## How lesson notebooks connect

**All hands-on notebooks that read ADLS** (`01-Day1-...`, `01-Day2-...`, `01-Day3-...`, `03-Day3-...`, `01-Day4-...`, `03-Day4-...`, `01-Day5-...`, `03-Day5-...`, `04-Day5-...`) must start with **`%run ./02-Mount-Azure-Data-Lake`** (typically the **first code cell**) so **`spark.conf` OAuth** and **`base_path`** are set on the cluster.

- Keep **`02-Mount-Azure-Data-Lake.ipynb`** in the **same workspace folder** as the lesson notebook, or adjust the path (e.g. `%run ../notebooks/02-Mount-Azure-Data-Lake`).
- After a **new cluster** or **RESTART**, run **`%run`** again, then the lesson **paths** cell (`BASE_PATH = base_path`, etc.).
- You can **Run all** on this notebook alone for debugging; in class it is normally executed **only** via `%run` from other notebooks.

## Step 1 — Storage account and OAuth (Service Principal)

OAuth configured for account: sadbtrng19032026


## Step 2 — Base path (ABFSS) and optional smoke read

base_path: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data


+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
# Paths from %run
BASE_PATH = base_path
DAY03_SILVER = BASE_PATH + "/day03/silver/flights_clean"
DAY03_GOLD   = BASE_PATH + "/day03/gold/flights_by_destination"

# Unity Catalog naming — CHANGE STUDENT_ID to your assigned u01–u16
CATALOG    = "shared"
STUDENT_ID = "u01"
SCHEMA_NAME = f"day04_{STUDENT_ID}_lab"
FULL_SCHEMA = f"{CATALOG}.{SCHEMA_NAME}"

print("Delta Silver :", DAY03_SILVER)
print("Delta Gold   :", DAY03_GOLD)
print("UC Schema    :", FULL_SCHEMA)

Delta Silver : abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day03/silver/flights_clean
Delta Gold   : abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day03/gold/flights_by_destination
UC Schema    : shared.day04_u01_lab


## 2 — Three-level namespace deep dive

Unity Catalog organizes every securable object in a **catalog → schema → object** hierarchy.

```
┌──────────────────────────────────────────────────────────┐
│  METASTORE  (regional — shared by workspaces)            │
│  ├── catalog: main                                       │
│  │   ├── schema: day04_u01_lab                           │
│  │   │   ├── VIEW  flights_silver_v                      │
│  │   │   ├── VIEW  flights_gold_v                        │
│  │   │   └── VIEW  flights_silver_dest_only_v            │
│  │   └── schema: default                                 │
│  ├── catalog: hive_metastore  (legacy)                   │
│  └── catalog: system          (metadata & audit)         │
└──────────────────────────────────────────────────────────┘
```

### Naming rules

| Level | Maps to | Naming in this lab |
|-------|---------|--------------------|
| **Catalog** | Environment / org boundary | `main` (or the catalog name your organization uses) |
| **Schema** | Database / project area | `day04_uXX_lab` (personal) |
| **Object** | Table, view, volume, function, model | `flights_silver_v`, etc. |

### Fully qualified name

```sql
SELECT * FROM main.day04_u01_lab.flights_silver_v
--           ^^^^  ^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^
--           cat   schema          object
```

### 2b — `main` vs `hive_metastore` vs `system`

| Catalog | Purpose | You use it for |
|---------|---------|----------------|
| **`main`** | Default UC catalog; new objects go here | Creating schemas, views, tables |
| **`hive_metastore`** | Legacy; pre-UC tables live here | Backward compat; avoid for new work |
| **`system`** | Databricks system tables (audit, lineage, billing) | Querying `information_schema`, audit logs |

**Rule:** Unless told otherwise, always create objects in **`main`** (or your org's named catalog).  
**Warning:** Mixing UC + legacy `hive_metastore` objects in the same query may produce unexpected behavior.

In [0]:
FULL_SCHEMA

'main.day04_u01_lab'

In [0]:
# Create your personal schema (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA} COMMENT 'Day 4 hands-on lab — UC fundamentals'")
print("✓ Schema ready:", FULL_SCHEMA)

✓ Schema ready: shared.day04_u01_lab


In [0]:
# Explore available catalogs
try:
    print("=== CATALOGS ===")
    spark.sql("SHOW CATALOGS").show(truncate=False)
except Exception as e:
    print("SHOW CATALOGS:", type(e).__name__, e)

=== CATALOGS ===
+---------------------------+
|catalog                    |
+---------------------------+
|databricks_training_premium|
|samples                    |
|shared                     |
|system                     |
+---------------------------+



In [0]:
# Schemas in your catalog — spot your new schema
try:
    print(f"=== SCHEMAS IN {CATALOG} ===")
    spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show(80, truncate=False)
except Exception as e:
    print("SHOW SCHEMAS:", type(e).__name__, e)

=== SCHEMAS IN shared ===
+------------------+
|databaseName      |
+------------------+
|day04_u01_lab     |
|default           |
|information_schema|
+------------------+



In [0]:
# USE SCHEMA shortcut (sets default for unqualified names in this session)
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    print(f"Default catalog.schema = {CATALOG}.{SCHEMA_NAME}")
except Exception as e:
    print("USE CATALOG/SCHEMA:", type(e).__name__, e)

Default catalog.schema = shared.day04_u01_lab


## 3 — Register governed views over Day 3 Delta (ABFS)

### Why views, not tables?

| Approach | Pros | Cons |
|----------|------|------|
| **View on `delta.\`path\``** | No data copy; each student gets a governed `GRANT` target; same physical files | Slower than a managed table (reads raw path each time) |
| **External TABLE** | Registered in metastore with stats | Duplicate `LOCATION` blocked for shared paths; needs external location + admin |
| **Managed TABLE** | Full UC governance, stats, optimize | Copies data; more storage cost |

This lab uses **views** because the class **shares** one ADLS path.

### 3a — Silver view (full projection)

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_v")
spark.sql(
    f"CREATE VIEW {FULL_SCHEMA}.flights_silver_v "
    f"COMMENT 'Day 3 Silver — all columns, cleaned flight routes' "
    f"AS SELECT * FROM delta.`{DAY03_SILVER}`"
)
print("Created:", f"{FULL_SCHEMA}.flights_silver_v")
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_v LIMIT 5").show(truncate=False)

Created: shared.day04_u01_lab.flights_silver_v
+------------------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|DEST_COUNTRY_NAME       |ORIGIN_COUNTRY_NAME|count|ingestion_ts              |source_file                                                                  |source_system|route_key                                                       |
+------------------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|Ireland                 |United States      |241  |2026-03-21 08:10:08.396536|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|day3_lab_csv |010ebd84c7788fa5ae2dd2fdc47625d5bed2fae7c83ee547290828ef3019b17f|
|Turk

### 3b — Gold view

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_gold_v")
spark.sql(
    f"CREATE VIEW {FULL_SCHEMA}.flights_gold_v "
    f"COMMENT 'Day 3 Gold — total flights by destination' "
    f"AS SELECT * FROM delta.`{DAY03_GOLD}`"
)
print("Created:", f"{FULL_SCHEMA}.flights_gold_v")
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_gold_v ORDER BY total_flights DESC LIMIT 5").show(truncate=False)

Created: shared.day04_u01_lab.flights_gold_v
+-----------------+-------------+
|DEST_COUNTRY_NAME|total_flights|
+-----------------+-------------+
|United States    |384932       |
|Canada           |8271         |
|Mexico           |6200         |
|United Kingdom   |1629         |
|Germany          |1392         |
+-----------------+-------------+



### 3c — Sanity checks (row counts)

In [0]:
silver_n = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_SCHEMA}.flights_silver_v").collect()[0]["n"]
gold_n   = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_SCHEMA}.flights_gold_v").collect()[0]["n"]
print(f"Silver rows: {silver_n}")
print(f"Gold rows:   {gold_n}")
assert silver_n > 0, "Silver is empty — check Day 3 paths"
assert gold_n > 0,   "Gold is empty — check Day 3 paths" 

Silver rows: 265
Gold rows:   125


### 3d — Why `SELECT *` in the view? Discussion

`SELECT *` passes through **all** columns from the Delta path. This is fine for:
- **Silver** (cleaned, not PII-heavy in this lab)
- **Gold** (aggregates — no individual records)

**In production**, prefer **explicit column lists** to:
1. Document the **contract** between view and consumers
2. Avoid exposing columns added later (schema evolution leakage)
3. Enable **column-level lineage** (UC tracks selected columns)

### 3e — **Projection view** (column minimization — governance pattern)

From **M-1/M-7 RBAC** labs: not every consumer needs **all** columns. A view can expose only safe fields.

**Scenario:** An analyst team only needs destination names and flight counts — not origin country or route keys.

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_dest_only_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_dest_only_v "
    + f"COMMENT 'Projection: destination + count only (column minimization)' "
    + f"AS SELECT DEST_COUNTRY_NAME, count FROM delta.`{DAY03_SILVER}`"
)
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_dest_only_v LIMIT 8").show(truncate=False)
print("Columns:", spark.table(f"{FULL_SCHEMA}.flights_silver_dest_only_v").columns)

+--------------------------------+-----+
|DEST_COUNTRY_NAME               |count|
+--------------------------------+-----+
|Ireland                         |241  |
|Turks and Caicos Islands        |146  |
|Marshall Islands                |87   |
|Spain                           |432  |
|Saudi Arabia                    |52   |
|Paraguay                        |100  |
|Saint Vincent and the Grenadines|11   |
|United Kingdom                  |1639 |
+--------------------------------+-----+

Columns: ['DEST_COUNTRY_NAME', 'count']


### 3f — **Filtered view** (row-level restriction pattern)

**Scenario:** A US-focused team should only see flights **to** the United States.

This is a **simple** version of Row-Level Security — a view with a `WHERE` clause.

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_us_only_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_us_only_v "
    + f"COMMENT 'Row filter: only US destinations (RLS lite)' "
    + f"AS SELECT * FROM delta.`{DAY03_SILVER}` WHERE DEST_COUNTRY_NAME = 'United States'"
)
us_count = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_SCHEMA}.flights_silver_us_only_v").collect()[0]["n"]
print(f"US-only view rows: {us_count}")
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_us_only_v LIMIT 5").show(truncate=False)

US-only view rows: 131
+-----------------+------------------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME           |count|ingestion_ts              |source_file                                                                  |source_system|route_key                                                       |
+-----------------+------------------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|United States    |Federated States of Micronesia|60   |2026-03-21 08:10:08.396536|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|day3_lab_csv |1304e93d44bf579b30e4a2d4401069c70d7f17b6d815914cd9264e941ef34013|
|United State

### 3g — **Enriched / tagged view** (zone tagging)

**Scenario:** An enterprise catalog adds **constant columns** for zone, owner, or classification tags (useful in Catalog Explorer).

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_tagged_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_tagged_v "
    + f"COMMENT 'Enriched: zone tag + refresh timestamp' "
    + f"AS SELECT *, 'CURATED_SILVER' AS uc_zone_tag, current_timestamp() AS view_refreshed_at FROM delta.`{DAY03_SILVER}`"
)
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_tagged_v LIMIT 3").show(truncate=False)

+------------------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+--------------+--------------------------+
|DEST_COUNTRY_NAME       |ORIGIN_COUNTRY_NAME|count|ingestion_ts              |source_file                                                                  |source_system|route_key                                                       |uc_zone_tag   |view_refreshed_at         |
+------------------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+--------------+--------------------------+
|Ireland                 |United States      |241  |2026-03-21 08:10:08.396536|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|day3_l

### 3h — **Masked view** (Tata Lab 16 pattern — simplified)

**Scenario:** Origin country is considered sensitive. Analysts see `***` instead of the real value.

(Real UC masking uses `ALTER TABLE ... SET MASKING POLICY` — shown in notebook **03** as theory.)

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_masked_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_masked_v "
    + f"COMMENT 'Masked: origin country hidden' "
    + f"AS SELECT DEST_COUNTRY_NAME, '***' AS ORIGIN_COUNTRY_NAME, count FROM delta.`{DAY03_SILVER}`"
)
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_masked_v LIMIT 8").show(truncate=False)

+--------------------------------+-------------------+-----+
|DEST_COUNTRY_NAME               |ORIGIN_COUNTRY_NAME|count|
+--------------------------------+-------------------+-----+
|Ireland                         |***                |241  |
|Turks and Caicos Islands        |***                |146  |
|Marshall Islands                |***                |87   |
|Spain                           |***                |432  |
|Saudi Arabia                    |***                |52   |
|Paraguay                        |***                |100  |
|Saint Vincent and the Grenadines|***                |11   |
|United Kingdom                  |***                |1639 |
+--------------------------------+-------------------+-----+



### 3i — Summary of views created

| View | Pattern | Columns |
|------|---------|---------|
| `flights_silver_v` | Full projection | All Silver cols |
| `flights_gold_v` | Full projection | All Gold cols |
| `flights_silver_dest_only_v` | Column minimization | 2 cols only |
| `flights_silver_us_only_v` | Row filter (RLS lite) | All cols, US-dest only |
| `flights_silver_tagged_v` | Enriched (zone tag) | All + 2 tag cols |
| `flights_silver_masked_v` | Column masking | dest + masked origin + count |

Each demonstrates a **real governance pattern** you will encounter in production.

In [0]:
# Count all views in your schema
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)

+-------------+--------------------------+-----------+
|database     |tableName                 |isTemporary|
+-------------+--------------------------+-----------+
|day04_u01_lab|flights_gold_v            |false      |
|day04_u01_lab|flights_silver_dest_only_v|false      |
|day04_u01_lab|flights_silver_masked_v   |false      |
|day04_u01_lab|flights_silver_tagged_v   |false      |
|day04_u01_lab|flights_silver_us_only_v  |false      |
|day04_u01_lab|flights_silver_v          |false      |
+-------------+--------------------------+-----------+



## 4 — Physical vs logical: prove row parity

**Key insight:** The view is **metadata** in UC; the bytes live in **ADLS Delta**. Row counts should match.

In [0]:
from pyspark.sql.functions import col

for label, path, view in [
    ("Silver", DAY03_SILVER, f"{FULL_SCHEMA}.flights_silver_v"),
    ("Gold",   DAY03_GOLD,   f"{FULL_SCHEMA}.flights_gold_v"),
]:
    n_delta = spark.read.format("delta").load(path).count()
    n_view  = spark.sql(f"SELECT COUNT(*) AS n FROM {view}").collect()[0]["n"]
    match   = "✓ MATCH" if n_delta == n_view else "✗ MISMATCH"
    print(f"{label}: delta={n_delta}  view={n_view}  {match}")

Silver: delta=265  view=265  ✓ MATCH
Gold: delta=125  view=125  ✓ MATCH


In [0]:
# Schema comparison: Delta columns vs UC view columns
d_cols = set(spark.read.format("delta").load(DAY03_SILVER).columns)
v_cols = set(spark.table(f"{FULL_SCHEMA}.flights_silver_v").columns)
print("Delta cols:", sorted(d_cols))
print("View cols: ", sorted(v_cols))
print("Extra in Delta:", sorted(d_cols - v_cols))
print("Extra in view: ", sorted(v_cols - d_cols))

Delta cols: ['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count', 'ingestion_ts', 'route_key', 'source_file', 'source_system']
View cols:  ['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count', 'ingestion_ts', 'route_key', 'source_file', 'source_system']
Extra in Delta: []
Extra in view:  []


## 5 — Metadata exploration (`SHOW`, `DESCRIBE`, `DESCRIBE EXTENDED`)

SQL metadata commands are **your first debugging tool** when something goes wrong with a view or table.

In [0]:
# 5a — SHOW TABLES (all objects in schema)
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)

+-------------+--------------------------+-----------+
|database     |tableName                 |isTemporary|
+-------------+--------------------------+-----------+
|day04_u01_lab|flights_gold_v            |false      |
|day04_u01_lab|flights_silver_dest_only_v|false      |
|day04_u01_lab|flights_silver_masked_v   |false      |
|day04_u01_lab|flights_silver_tagged_v   |false      |
|day04_u01_lab|flights_silver_us_only_v  |false      |
|day04_u01_lab|flights_silver_v          |false      |
+-------------+--------------------------+-----------+



In [0]:
# 5b — DESCRIBE (column names + types)
spark.sql(f"DESCRIBE {FULL_SCHEMA}.flights_silver_v").show(truncate=False)

+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|DEST_COUNTRY_NAME  |string   |NULL   |
|ORIGIN_COUNTRY_NAME|string   |NULL   |
|count              |bigint   |NULL   |
|ingestion_ts       |timestamp|NULL   |
|source_file        |string   |NULL   |
|source_system      |string   |NULL   |
|route_key          |string   |NULL   |
+-------------------+---------+-------+



In [0]:
# 5c — DESCRIBE EXTENDED (ownership, view text, properties, etc.)
spark.sql(f"DESCRIBE EXTENDED {FULL_SCHEMA}.flights_silver_v").show(100, truncate=False)

+----------------------------+------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                         |comment|
+----------------------------+------------------------------------------------------------------------------------------------------------------+-------+
|DEST_COUNTRY_NAME           |string                                                                                                            |NULL   |
|ORIGIN_COUNTRY_NAME         |string                                                                                                            |NULL   |
|count                       |bigint                                                                                                            |NULL   |
|ingestion_ts                |timestamp                                     

In [0]:
# 5d — DESCRIBE on projection view — notice fewer columns
spark.sql(f"DESCRIBE {FULL_SCHEMA}.flights_silver_dest_only_v").show(truncate=False)

+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|DEST_COUNTRY_NAME|string   |NULL   |
|count            |bigint   |NULL   |
+-----------------+---------+-------+



In [0]:
# 5e — DESCRIBE EXTENDED on Gold
spark.sql(f"DESCRIBE EXTENDED {FULL_SCHEMA}.flights_gold_v").show(100, truncate=False)

+----------------------------+--------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                           |comment|
+----------------------------+--------------------------------------------------------------------------------------------------------------------+-------+
|DEST_COUNTRY_NAME           |string                                                                                                              |NULL   |
|total_flights               |bigint                                                                                                              |NULL   |
|                            |                                                                                                                    |       |
|# Detailed Table Information|                                  

In [0]:
# 5f — SHOW CREATE VIEW (view definition SQL — varies by DBR)
for v in ["flights_silver_v", "flights_silver_dest_only_v", "flights_silver_masked_v"]:
    try:
        print(f"=== {v} ===")
        spark.sql(f"SHOW CREATE TABLE {FULL_SCHEMA}.{v}").show(truncate=False)
    except Exception as e:
        print(f"  SHOW CREATE: {type(e).__name__}")

=== flights_silver_v ===
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|createtab_stmt                                                                                                                                                                                                                                                                                                                                                                  |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 6 — Spark catalog API (programmatic discovery)

The `spark.catalog` Python API lets you **list** databases, tables, and columns without writing SQL — useful in orchestration and testing code.

In [0]:
# 6a — List catalogs / databases visible to this session
try:
    dbs = spark.catalog.listDatabases()
    for d in dbs[:20]:
        print(d.name, "|", getattr(d, "description", ""))
except Exception as e:
    print("listDatabases:", type(e).__name__, e)

day04_u01_lab | Day 4 hands-on lab — UC fundamentals
default | Default schema (auto-created)
information_schema | Information schema (auto-created)


In [0]:
# 6b — List tables in your schema
try:
    for t in spark.catalog.listTables(SCHEMA_NAME):
        print(f"  {t.name:40s} | type={getattr(t, 'tableType', '?')}")
except Exception as e:
    print("listTables:", type(e).__name__, e)
    print("Fallback → SHOW TABLES SQL")

  flights_gold_v                           | type=VIEW
  flights_silver_dest_only_v               | type=VIEW
  flights_silver_masked_v                  | type=VIEW
  flights_silver_tagged_v                  | type=VIEW
  flights_silver_us_only_v                 | type=VIEW
  flights_silver_v                         | type=VIEW


In [0]:
# 6c — spark.table() = DataFrame from a UC name
silver_df = spark.table(f"{FULL_SCHEMA}.flights_silver_dest_only_v")
print("columns:", silver_df.columns)
print("count:", silver_df.count())
silver_df.show(5, truncate=False)

columns: ['DEST_COUNTRY_NAME', 'count']
count: 265
+------------------------+-----+
|DEST_COUNTRY_NAME       |count|
+------------------------+-----+
|Ireland                 |241  |
|Turks and Caicos Islands|146  |
|Marshall Islands        |87   |
|Spain                   |432  |
|Saudi Arabia            |52   |
+------------------------+-----+
only showing top 5 rows


## 7 — Analytics SQL on governed views (10 queries)

These queries use **only** `main.day04_uXX_lab.*` names — identical to how an analyst would work post-`GRANT`.

### 7a — Top 10 destinations by total flights (Silver)

In [0]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, SUM(count) AS flights FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").show(truncate=False)

+------------------+-------+
|DEST_COUNTRY_NAME |flights|
+------------------+-------+
|United States     |386400 |
|Canada            |8281   |
|Mexico            |6210   |
|Germany           |1772   |
|United Kingdom    |1639   |
|Japan             |1393   |
|Dominican Republic|1119   |
|France            |1034   |
|Brazil            |1005   |
|The Bahamas       |913    |
+------------------+-------+



### 7b — Top 10 origin countries (Silver)

In [0]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS flights FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").show(truncate=False)

+-------------------+-------+
|ORIGIN_COUNTRY_NAME|flights|
+-------------------+-------+
|United States      |386702 |
|Canada             |8316   |
|Mexico             |6231   |
|United Kingdom     |1514   |
|Germany            |1417   |
|Japan              |1318   |
|Dominican Republic |1161   |
|The Bahamas        |970    |
|Colombia           |843    |
|France             |787    |
+-------------------+-------+



### 7c — US–US route (domestic flights)

In [0]:
spark.sql(f"""SELECT * FROM {FULL_SCHEMA}.flights_silver_v WHERE DEST_COUNTRY_NAME = 'United States' AND ORIGIN_COUNTRY_NAME = 'United States'""").show(truncate=False)

+-----------------+-------------------+------+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count |ingestion_ts              |source_file                                                                  |source_system|route_key                                                       |
+-----------------+-------------------+------+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|United States    |United States      |348124|2026-03-21 08:10:08.396536|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|day3_lab_csv |39e72e7731802c1a3872516516d6ebbc292f77b7ffb1c8170dc476d55ba06150|
+-----------------+-------------------+------+--------------------------+---

### 7d — Gold: max vs median total_flights

In [0]:
spark.sql(f"""SELECT MAX(total_flights) AS max_flights, percentile_approx(total_flights, 0.5) AS median_flights, AVG(total_flights) AS avg_flights FROM {FULL_SCHEMA}.flights_gold_v""").show(truncate=False)

+-----------+--------------+-----------+
|max_flights|median_flights|avg_flights|
+-----------+--------------+-----------+
|384932     |62            |3378.152   |
+-----------+--------------+-----------+



### 7e — High-volume destinations (HAVING)

In [0]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, SUM(count) AS s FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 HAVING SUM(count) > 1000 ORDER BY 2 DESC LIMIT 15""").show(truncate=False)

+------------------+------+
|DEST_COUNTRY_NAME |s     |
+------------------+------+
|United States     |386400|
|Canada            |8281  |
|Mexico            |6210  |
|Germany           |1772  |
|United Kingdom    |1639  |
|Japan             |1393  |
|Dominican Republic|1119  |
|France            |1034  |
|Brazil            |1005  |
+------------------+------+



### 7f — Distinct destination count

In [0]:
spark.sql(f"""SELECT COUNT(DISTINCT DEST_COUNTRY_NAME) AS distinct_dest FROM {FULL_SCHEMA}.flights_silver_v""").show(truncate=False)

+-------------+
|distinct_dest|
+-------------+
|127          |
+-------------+



### 7g — Destinations starting with 'C'

In [0]:
spark.sql(f"""SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v WHERE DEST_COUNTRY_NAME LIKE 'C%' ORDER BY 1""").show(truncate=False)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
|Canada           |
|Cape Verde       |
|Cayman Islands   |
|Chile            |
|China            |
|Chine            |
|Colombia         |
|Cook Islands     |
|Costa Rica       |
|Cuba             |
|Curacao          |
|Cyprus           |
|Czech Republic   |
+-----------------+



### 7h — CASE bucket: high / medium / low volume routes

In [0]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count, CASE WHEN count >= 1000 THEN 'high' WHEN count >= 100 THEN 'med' ELSE 'low' END AS bucket FROM {FULL_SCHEMA}.flights_silver_v LIMIT 20""").show(truncate=False)

+--------------------------------+-------------------+-----+------+
|DEST_COUNTRY_NAME               |ORIGIN_COUNTRY_NAME|count|bucket|
+--------------------------------+-------------------+-----+------+
|Ireland                         |United States      |241  |med   |
|Turks and Caicos Islands        |United States      |146  |med   |
|Marshall Islands                |United States      |87   |low   |
|Spain                           |United States      |432  |med   |
|Saudi Arabia                    |United States      |52   |low   |
|Paraguay                        |United States      |100  |med   |
|Saint Vincent and the Grenadines|United States      |11   |low   |
|United Kingdom                  |United States      |1639 |high  |
|Cyprus                          |United States      |12   |low   |
|Israel                          |United States      |127  |med   |
|Ecuador                         |United States      |282  |med   |
|Afghanistan                     |United States 

### 7i — Routes with count between 100 and 500

In [0]:
spark.sql(f"""SELECT * FROM {FULL_SCHEMA}.flights_silver_v WHERE count BETWEEN 100 AND 500 ORDER BY count DESC LIMIT 15""").show(truncate=False)

+-----------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|ingestion_ts              |source_file                                                                  |source_system|route_key                                                       |
+-----------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|Costa Rica       |United States      |487  |2026-03-21 08:10:08.396536|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|day3_lab_csv |58c6dce2ba224bd6446747d4f84a5a72df3c3006d5a2e01d678ab87f5a036483|
|United States    |El Salvador        |475  |2026-03-21 08:10:08.396536|abfss://

### 7j — US-only view: top 5 origin countries to US

In [0]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS to_us FROM {FULL_SCHEMA}.flights_silver_us_only_v GROUP BY 1 ORDER BY 2 DESC LIMIT 5""").show(truncate=False)

+-------------------+------+
|ORIGIN_COUNTRY_NAME|to_us |
+-------------------+------+
|United States      |348124|
|Canada             |8316  |
|Mexico             |6231  |
|United Kingdom     |1514  |
|Germany            |1417  |
+-------------------+------+



## 8 — DataFrame exercises on governed views

Same data, PySpark API. Privilege checks still apply (UC validates the name).

In [0]:
from pyspark.sql.functions import col, sum as fsum, avg as favg, count as fcount, desc

# 8a — Gold: top 12 destinations
gdf = spark.table(f"{FULL_SCHEMA}.flights_gold_v")
gdf.orderBy(desc("total_flights")).select("DEST_COUNTRY_NAME", "total_flights").show(12, truncate=False)

+------------------+-------------+
|DEST_COUNTRY_NAME |total_flights|
+------------------+-------------+
|United States     |384932       |
|Canada            |8271         |
|Mexico            |6200         |
|United Kingdom    |1629         |
|Germany           |1392         |
|Japan             |1383         |
|Dominican Republic|1109         |
|Brazil            |995          |
|The Bahamas       |903          |
|Colombia          |785          |
|France            |774          |
|Jamaica           |733          |
+------------------+-------------+
only showing top 12 rows


In [0]:
# 8b — Silver: outbound flights per origin
spark.table(f"{FULL_SCHEMA}.flights_silver_v") \
    .groupBy("ORIGIN_COUNTRY_NAME") \
    .agg(fsum("count").alias("outbound")) \
    .orderBy(desc("outbound")) \
    .show(10, truncate=False)

+-------------------+--------+
|ORIGIN_COUNTRY_NAME|outbound|
+-------------------+--------+
|United States      |386702  |
|Canada             |8316    |
|Mexico             |6231    |
|United Kingdom     |1514    |
|Germany            |1417    |
|Japan              |1318    |
|Dominican Republic |1161    |
|The Bahamas        |970     |
|Colombia           |843     |
|France             |787     |
+-------------------+--------+
only showing top 10 rows


In [0]:
# 8c — Projection view: average count per destination
spark.table(f"{FULL_SCHEMA}.flights_silver_dest_only_v") \
    .groupBy("DEST_COUNTRY_NAME") \
    .agg(favg("count").alias("avg_count"), fcount("count").alias("routes")) \
    .orderBy(desc("avg_count")) \
    .show(12, truncate=False)

+------------------+-----------------+------+
|DEST_COUNTRY_NAME |avg_count        |routes|
+------------------+-----------------+------+
|Canada            |8281.0           |1     |
|Mexico            |6210.0           |1     |
|United States     |2949.618320610687|131   |
|United Kingdom    |1639.0           |1     |
|Japan             |1393.0           |1     |
|Dominican Republic|1119.0           |1     |
|Brazil            |1005.0           |1     |
|The Bahamas       |913.0            |1     |
|Colombia          |795.0            |1     |
|Jamaica           |743.0            |1     |
|South Korea       |693.0            |1     |
|Netherlands       |596.0            |1     |
+------------------+-----------------+------+
only showing top 12 rows


In [0]:
# 8d — Tagged view: check enrichment columns
spark.table(f"{FULL_SCHEMA}.flights_silver_tagged_v") \
    .select("DEST_COUNTRY_NAME", "uc_zone_tag", "view_refreshed_at") \
    .limit(5).show(truncate=False)

+------------------------+--------------+--------------------------+
|DEST_COUNTRY_NAME       |uc_zone_tag   |view_refreshed_at         |
+------------------------+--------------+--------------------------+
|Ireland                 |CURATED_SILVER|2026-03-21 13:40:49.710214|
|Turks and Caicos Islands|CURATED_SILVER|2026-03-21 13:40:49.710214|
|Marshall Islands        |CURATED_SILVER|2026-03-21 13:40:49.710214|
|Spain                   |CURATED_SILVER|2026-03-21 13:40:49.710214|
|Saudi Arabia            |CURATED_SILVER|2026-03-21 13:40:49.710214|
+------------------------+--------------+--------------------------+



In [0]:
# 8e — Masked view: prove origin is hidden
spark.table(f"{FULL_SCHEMA}.flights_silver_masked_v") \
    .select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count") \
    .show(8, truncate=False)

+--------------------------------+-------------------+-----+
|DEST_COUNTRY_NAME               |ORIGIN_COUNTRY_NAME|count|
+--------------------------------+-------------------+-----+
|Ireland                         |***                |241  |
|Turks and Caicos Islands        |***                |146  |
|Marshall Islands                |***                |87   |
|Spain                           |***                |432  |
|Saudi Arabia                    |***                |52   |
|Paraguay                        |***                |100  |
|Saint Vincent and the Grenadines|***                |11   |
|United Kingdom                  |***                |1639 |
+--------------------------------+-------------------+-----+
only showing top 8 rows


## 9 — Joins, subqueries, window functions (governed views)

### 9a — Join Silver + Gold on destination

In [0]:
spark.sql(f"""
SELECT s.DEST_COUNTRY_NAME, SUM(s.count) AS silver_sum, MAX(g.total_flights) AS gold_total
FROM {FULL_SCHEMA}.flights_silver_v s
JOIN {FULL_SCHEMA}.flights_gold_v g ON s.DEST_COUNTRY_NAME = g.DEST_COUNTRY_NAME
GROUP BY s.DEST_COUNTRY_NAME
ORDER BY silver_sum DESC LIMIT 12
""").show(truncate=False)

+------------------+----------+----------+
|DEST_COUNTRY_NAME |silver_sum|gold_total|
+------------------+----------+----------+
|United States     |386400    |384932    |
|Canada            |8281      |8271      |
|Mexico            |6210      |6200      |
|Germany           |1772      |1392      |
|United Kingdom    |1639      |1629      |
|Japan             |1393      |1383      |
|Dominican Republic|1119      |1109      |
|France            |1034      |774       |
|Brazil            |1005      |995       |
|The Bahamas       |913       |903       |
|Colombia          |795       |785       |
|Jamaica           |743       |733       |
+------------------+----------+----------+



### 9b — Subquery EXISTS (routes to high-volume destinations)

In [0]:
spark.sql(f"""
SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count
FROM {FULL_SCHEMA}.flights_silver_v s
WHERE EXISTS (
  SELECT 1 FROM {FULL_SCHEMA}.flights_gold_v g
  WHERE g.DEST_COUNTRY_NAME = s.DEST_COUNTRY_NAME AND g.total_flights > 5000
)
ORDER BY count DESC LIMIT 20
""").show(truncate=False)

+-----------------+-------------------+------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count |
+-----------------+-------------------+------+
|United States    |United States      |348124|
|United States    |Canada             |8316  |
|Canada           |United States      |8281  |
|United States    |Mexico             |6231  |
|Mexico           |United States      |6210  |
|United States    |United Kingdom     |1514  |
|United States    |Germany            |1417  |
|United States    |Japan              |1318  |
|United States    |Dominican Republic |1161  |
|United States    |The Bahamas        |970   |
|United States    |Colombia           |843   |
|United States    |France             |787   |
|United States    |Jamaica            |769   |
|United States    |South Korea        |632   |
|United States    |Brazil             |589   |
|United States    |Netherlands        |581   |
|United States    |China              |516   |
|United States    |Costa Rica         |514   |
|United State

### 9c — Window: rank destinations by total_flights

In [0]:
spark.sql(f"""
SELECT DEST_COUNTRY_NAME, total_flights,
       ROW_NUMBER() OVER (ORDER BY total_flights DESC) AS rnk,
       NTILE(4) OVER (ORDER BY total_flights DESC) AS quartile
FROM {FULL_SCHEMA}.flights_gold_v
""").where("rnk <= 15").show(truncate=False)

+------------------+-------------+---+--------+
|DEST_COUNTRY_NAME |total_flights|rnk|quartile|
+------------------+-------------+---+--------+
|United States     |384932       |1  |1       |
|Canada            |8271         |2  |1       |
|Mexico            |6200         |3  |1       |
|United Kingdom    |1629         |4  |1       |
|Germany           |1392         |5  |1       |
|Japan             |1383         |6  |1       |
|Dominican Republic|1109         |7  |1       |
|Brazil            |995          |8  |1       |
|The Bahamas       |903          |9  |1       |
|Colombia          |785          |10 |1       |
|France            |774          |11 |1       |
|Jamaica           |733          |12 |1       |
|South Korea       |683          |13 |1       |
|Netherlands       |586          |14 |1       |
|El Salvador       |519          |15 |1       |
+------------------+-------------+---+--------+



### 9d — Window: top 3 routes per destination (Silver)

In [0]:
spark.sql(f"""
SELECT * FROM (
  SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count,
         ROW_NUMBER() OVER (PARTITION BY DEST_COUNTRY_NAME ORDER BY count DESC) AS rnk
  FROM {FULL_SCHEMA}.flights_silver_v
) t WHERE rnk <= 3
ORDER BY DEST_COUNTRY_NAME, rnk
LIMIT 40
""").show(truncate=False)

+---------------------------------+-------------------+-----+---+
|DEST_COUNTRY_NAME                |ORIGIN_COUNTRY_NAME|count|rnk|
+---------------------------------+-------------------+-----+---+
|Afghanistan                      |United States      |21   |1  |
|Angola                           |United States      |24   |1  |
|Anguilla                         |United States      |31   |1  |
|Antigua and Barbuda              |United States      |133  |1  |
|Argentina                        |United States      |194  |1  |
|Aruba                            |United States      |369  |1  |
|Australia                        |United States      |300  |1  |
|Austria                          |United States      |46   |1  |
|Azerbaijan                       |United States      |11   |1  |
|Bahrain                          |United States      |40   |1  |
|Barbados                         |United States      |140  |1  |
|Belgium                          |United States      |418  |1  |
|Belize   

### 9e — CTE: destinations above average Gold total_flights

In [0]:
spark.sql(f"""
WITH avg_tbl AS (SELECT AVG(total_flights) AS avg_f FROM {FULL_SCHEMA}.flights_gold_v)
SELECT g.DEST_COUNTRY_NAME, g.total_flights, ROUND(a.avg_f, 1) AS avg_flights
FROM {FULL_SCHEMA}.flights_gold_v g
CROSS JOIN avg_tbl a
WHERE g.total_flights > a.avg_f
ORDER BY g.total_flights DESC
""").show(truncate=False)

+-----------------+-------------+-----------+
|DEST_COUNTRY_NAME|total_flights|avg_flights|
+-----------------+-------------+-----------+
|United States    |384932       |3378.2     |
|Canada           |8271         |3378.2     |
|Mexico           |6200         |3378.2     |
+-----------------+-------------+-----------+



### 9f — INTERSECT: destinations present in both Silver and Gold

In [0]:
spark.sql(f"""
SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v
INTERSECT
SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_gold_v
ORDER BY 1 LIMIT 25
""").show(truncate=False)

+---------------------------------+
|DEST_COUNTRY_NAME                |
+---------------------------------+
|Afghanistan                      |
|Angola                           |
|Anguilla                         |
|Antigua and Barbuda              |
|Argentina                        |
|Aruba                            |
|Australia                        |
|Austria                          |
|Azerbaijan                       |
|Bahrain                          |
|Barbados                         |
|Belgium                          |
|Belize                           |
|Bermuda                          |
|Bolivia                          |
|Bonaire, Sint Eustatius, and Saba|
|Brazil                           |
|British Virgin Islands           |
|Bulgaria                         |
|Canada                           |
+---------------------------------+
only showing top 20 rows


### 9g — LEFT ANTI JOIN: Gold destinations NOT in Silver (should be empty or rare)

In [0]:
spark.sql(f"""
SELECT g.DEST_COUNTRY_NAME
FROM {FULL_SCHEMA}.flights_gold_v g
LEFT ANTI JOIN {FULL_SCHEMA}.flights_silver_v s
  ON g.DEST_COUNTRY_NAME = s.DEST_COUNTRY_NAME
LIMIT 20
""").show(truncate=False)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
+-----------------+



## 10 — `EXPLAIN` (query plan for a governed view)

Check whether **predicate pushdown** reaches the Delta scan layer (look for `PushedFilters`).

In [0]:
spark.sql(f"EXPLAIN EXTENDED SELECT * FROM {FULL_SCHEMA}.flights_silver_v WHERE DEST_COUNTRY_NAME = 'Germany'").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Compare: projection view should scan fewer columns
spark.sql(f"EXPLAIN SELECT * FROM {FULL_SCHEMA}.flights_silver_dest_only_v WHERE DEST_COUNTRY_NAME = 'Germany'").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                      

## 12 — Temp view vs permanent UC view

| Kind | Lifetime | `GRANT`-able? |
|------|----------|---------------|
| **`CREATE TEMPORARY VIEW`** | Spark session only | No |
| **`CREATE VIEW` in UC** | Metastore-persisted | Yes |

In [0]:
# Temp view: useful for ad-hoc exploration, not for sharing
spark.table(f"{FULL_SCHEMA}.flights_silver_v").where("count > 500").createOrReplaceTempView("tmp_high_vol")
spark.sql("SELECT COUNT(*) AS n FROM tmp_high_vol").show()
spark.sql("SELECT * FROM tmp_high_vol ORDER BY count DESC LIMIT 6").show(truncate=False)

+---+
|  n|
+---+
| 30|
+---+

+-----------------+-------------------+------+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count |ingestion_ts              |source_file                                                                  |source_system|route_key                                                       |
+-----------------+-------------------+------+--------------------------+-----------------------------------------------------------------------------+-------------+----------------------------------------------------------------+
|United States    |United States      |348124|2026-03-21 08:10:08.396536|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|day3_lab_csv |39e72e7731802c1a3872516516d6ebbc292f77b7ffb1c8170dc476d55ba06150|
|United States    |Canada             |8316  

In [0]:
# Confirm temp view is NOT in UC
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)
# tmp_high_vol does NOT appear — it lives in the session only

+-------------+--------------------------+-----------+
|database     |tableName                 |isTemporary|
+-------------+--------------------------+-----------+
|day04_u01_lab|flights_gold_v            |false      |
|day04_u01_lab|flights_silver_dest_only_v|false      |
|day04_u01_lab|flights_silver_masked_v   |false      |
|day04_u01_lab|flights_silver_tagged_v   |false      |
|day04_u01_lab|flights_silver_us_only_v  |false      |
|day04_u01_lab|flights_silver_v          |false      |
|             |tmp_high_vol              |true       |
+-------------+--------------------------+-----------+



## 13 — `COMMENT ON VIEW` (documentation in Catalog Explorer)

In [0]:
for v, cmnt in [
    ("flights_gold_v", "Gold KPI by destination — Day 3 pipeline; Day 4 UC view."),
    ("flights_silver_us_only_v", "Row-filtered view: only US destinations (RLS lite)."),
]:
    try:
        spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.{v} IS '{cmnt}'")
        print(f"Comment set on {v}")
    except Exception as e:
        print(f"COMMENT ON {v}:", type(e).__name__)

Comment set on flights_gold_v
Comment set on flights_silver_us_only_v


## 14 — Delta storage metadata (`DESCRIBE DETAIL`, `DESCRIBE HISTORY`)

The **physical** Delta table behind the view has its own metadata. These commands work on the **path**, not the view name.

In [0]:
# DESCRIBE DETAIL — file count, size, partitioning
for label, path in [("Silver", DAY03_SILVER), ("Gold", DAY03_GOLD)]:
    print(f"=== {label} DETAIL ===")
    spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select("format", "numFiles", "sizeInBytes", "partitionColumns").show(truncate=False)

=== Silver DETAIL ===
+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |1       |23685      |[]              |
+------+--------+-----------+----------------+

=== Gold DETAIL ===
+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |1       |2807       |[]              |
+------+--------+-----------+----------------+



In [0]:
# DESCRIBE HISTORY — transaction log
spark.sql(f"DESCRIBE HISTORY delta.`{DAY03_SILVER}`").select("version", "timestamp", "operation").show(10, truncate=False)

+-------+-------------------+------------+
|version|timestamp          |operation   |
+-------+-------------------+------------+
|50     |2026-03-21 09:27:12|VACUUM END  |
|49     |2026-03-21 09:27:07|VACUUM START|
|48     |2026-03-21 09:26:29|VACUUM END  |
|47     |2026-03-21 09:26:25|VACUUM START|
|46     |2026-03-21 09:25:20|VACUUM END  |
|45     |2026-03-21 09:25:16|VACUUM START|
|44     |2026-03-21 09:15:31|VACUUM END  |
|43     |2026-03-21 09:15:27|VACUUM START|
|42     |2026-03-21 09:09:18|VACUUM END  |
|41     |2026-03-21 09:09:13|VACUUM START|
+-------+-------------------+------------+
only showing top 10 rows


In [0]:
# Extract numFiles programmatically
nf = spark.sql(f"DESCRIBE DETAIL delta.`{DAY03_GOLD}`").select("numFiles").collect()[0]["numFiles"]
print(f"Gold Delta numFiles = {nf}")

Gold Delta numFiles = 1


## 15 — Mini challenges

Complete these in the empty code cells. Compare approaches with others in the class or review in a follow-up session.

### C1 — Write a CTE that ranks Gold destinations and returns only **top 5** with `ROW_NUMBER()`.

In [0]:
# Your solution:


### C2 — Count **distinct** `ORIGIN_COUNTRY_NAME` in Silver using `COUNT(DISTINCT ...)`.

In [0]:
# Your solution:


### C3 — Join `flights_silver_us_only_v` with `flights_gold_v` on `DEST_COUNTRY_NAME` — does `United States` appear?

In [0]:
# Your solution:


### C4 — Using `DESCRIBE DETAIL` on `DAY03_GOLD`, extract `sizeInBytes` into a Python variable and print it in MB.

In [0]:
# Your solution:


### C5 — Explain in one sentence: **Why can two students both `SELECT` the same ADLS path but still need different UC schemas?**

In [0]:
# Your solution:


### C6 — Create a view `flights_gold_top10_v` that only exposes the **top 10** destinations (use a subquery or CTE).

In [0]:
# Your solution:


## 16 — SQL skill builders (10 drills — run each cell, tweak filters)

### 16a — Average count per destination (top 12)

In [0]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, ROUND(AVG(count),1) AS avg_cnt FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 12""").show(truncate=False)

+------------------+-------+
|DEST_COUNTRY_NAME |avg_cnt|
+------------------+-------+
|Canada            |8281.0 |
|Mexico            |6210.0 |
|United States     |2949.6 |
|United Kingdom    |1639.0 |
|Japan             |1393.0 |
|Dominican Republic|1119.0 |
|Brazil            |1005.0 |
|The Bahamas       |913.0  |
|Colombia          |795.0  |
|Jamaica           |743.0  |
|South Korea       |693.0  |
|Netherlands       |596.0  |
+------------------+-------+



### 16b — Median count per origin country

In [0]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, percentile_approx(count, 0.5) AS med FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 12""").show(truncate=False)

+-------------------+----+
|ORIGIN_COUNTRY_NAME|med |
+-------------------+----+
|Canada             |8316|
|Mexico             |6231|
|United Kingdom     |1514|
|Germany            |1417|
|Japan              |1318|
|Dominican Republic |1161|
|The Bahamas        |970 |
|Colombia           |843 |
|France             |787 |
|Jamaica            |769 |
|South Korea        |632 |
|Brazil             |589 |
+-------------------+----+



### 16c — Pair count: how many distinct (dest, origin) pairs?

In [0]:
spark.sql(f"""SELECT COUNT(*) AS pairs FROM (SELECT DISTINCT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v)""").show(truncate=False)

+-----+
|pairs|
+-----+
|259  |
+-----+



### 16d — Gold: destinations above 90th percentile

In [0]:
spark.sql(f"""SELECT * FROM {FULL_SCHEMA}.flights_gold_v WHERE total_flights >= (SELECT percentile_approx(total_flights, 0.9) FROM {FULL_SCHEMA}.flights_gold_v) ORDER BY total_flights DESC""").show(truncate=False)

+------------------+-------------+
|DEST_COUNTRY_NAME |total_flights|
+------------------+-------------+
|United States     |384932       |
|Canada            |8271         |
|Mexico            |6200         |
|United Kingdom    |1629         |
|Germany           |1392         |
|Japan             |1383         |
|Dominican Republic|1109         |
|Brazil            |995          |
|The Bahamas       |903          |
|Colombia          |785          |
|France            |774          |
|Jamaica           |733          |
|South Korea       |683          |
+------------------+-------------+



### 16e — Compare sum(Silver) vs sum(Gold)

In [0]:
spark.sql(f"""SELECT (SELECT SUM(count) FROM {FULL_SCHEMA}.flights_silver_v) AS silver_sum, (SELECT SUM(total_flights) FROM {FULL_SCHEMA}.flights_gold_v) AS gold_sum""").show(truncate=False)

+----------+--------+
|silver_sum|gold_sum|
+----------+--------+
|426038    |422269  |
+----------+--------+



### 16f — Silver: countries with exactly 1 route

In [0]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, COUNT(*) AS routes FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 HAVING COUNT(*) = 1 ORDER BY 1""").show(truncate=False)

+---------------------------------+------+
|DEST_COUNTRY_NAME                |routes|
+---------------------------------+------+
|Afghanistan                      |1     |
|Angola                           |1     |
|Anguilla                         |1     |
|Antigua and Barbuda              |1     |
|Argentina                        |1     |
|Aruba                            |1     |
|Australia                        |1     |
|Austria                          |1     |
|Azerbaijan                       |1     |
|Bahrain                          |1     |
|Barbados                         |1     |
|Belgium                          |1     |
|Belize                           |1     |
|Bermuda                          |1     |
|Bolivia                          |1     |
|Bonaire, Sint Eustatius, and Saba|1     |
|Brazil                           |1     |
|British Virgin Islands           |1     |
|Bulgaria                         |1     |
|Canada                           |1     |
+----------

### 16g — Gold: ratio of each destination to grand total

In [0]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, total_flights, ROUND(total_flights * 100.0 / (SELECT SUM(total_flights) FROM {FULL_SCHEMA}.flights_gold_v), 2) AS pct FROM {FULL_SCHEMA}.flights_gold_v ORDER BY pct DESC LIMIT 15""").show(truncate=False)

+------------------+-------------+-----+
|DEST_COUNTRY_NAME |total_flights|pct  |
+------------------+-------------+-----+
|United States     |384932       |91.16|
|Canada            |8271         |1.96 |
|Mexico            |6200         |1.47 |
|United Kingdom    |1629         |0.39 |
|Germany           |1392         |0.33 |
|Japan             |1383         |0.33 |
|Dominican Republic|1109         |0.26 |
|Brazil            |995          |0.24 |
|The Bahamas       |903          |0.21 |
|Colombia          |785          |0.19 |
|France            |774          |0.18 |
|Jamaica           |733          |0.17 |
|South Korea       |683          |0.16 |
|Netherlands       |586          |0.14 |
|El Salvador       |519          |0.12 |
+------------------+-------------+-----+



### 16h — Masked view: aggregate on masked data (prove it works)

In [0]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS s FROM {FULL_SCHEMA}.flights_silver_masked_v GROUP BY 1""").show(truncate=False)

+-------------------+------+
|ORIGIN_COUNTRY_NAME|s     |
+-------------------+------+
|***                |426038|
+-------------------+------+



### 16i — Silver: MIN, MAX, STDDEV of count

In [0]:
spark.sql(f"""SELECT MIN(count) AS min_c, MAX(count) AS max_c, ROUND(STDDEV(count),1) AS std_c FROM {FULL_SCHEMA}.flights_silver_v""").show(truncate=False)

+-----+------+-------+
|min_c|max_c |std_c  |
+-----+------+-------+
|11   |348124|21386.7|
+-----+------+-------+



### 16j — UNION destinations from Silver + Gold (deduplicated)

In [0]:
spark.sql(f"""SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v UNION SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_gold_v ORDER BY 1""").show(truncate=False)

+---------------------------------+
|DEST_COUNTRY_NAME                |
+---------------------------------+
|Afghanistan                      |
|Angola                           |
|Anguilla                         |
|Antigua and Barbuda              |
|Argentina                        |
|Aruba                            |
|Australia                        |
|Austria                          |
|Azerbaijan                       |
|Bahrain                          |
|Barbados                         |
|Belgium                          |
|Belize                           |
|Bermuda                          |
|Bolivia                          |
|Bonaire, Sint Eustatius, and Saba|
|Brazil                           |
|British Virgin Islands           |
|Bulgaria                         |
|Canada                           |
+---------------------------------+
only showing top 20 rows


## Next

Continue with `03-Day4-Unity-Catalog-Security-Lineage.ipynb` (grants, RBAC, metadata views, lineage, audit).

Cleanup (optional): drop your lab schema only if class policy allows.

```python
# spark.sql(f"DROP SCHEMA IF EXISTS {FULL_SCHEMA} CASCADE")
```